# WEEX V6.51 — MARKET i LIMIT
Twardy filtr strukturalnego TP1, wspolna historia OI, ZIP automatycznie.

In [ ]:
import requests
import shutil
from pathlib import Path
from google.colab import drive, files

# Wspolna historia OI wszystkich wersji
drive.mount("/content/drive")
oi_dir = Path("/content/drive/MyDrive/WEEX")
oi_dir.mkdir(parents=True, exist_ok=True)
state = oi_dir / "oi_history_shared.json"
if not state.exists():
    old = [oi_dir / f"weex_v{v}_oi_history_state.json" for v in (650,649,648,647,646,645,644)]
    present = [p for p in old if p.exists()]
    if present:
        shutil.copy2(max(present, key=lambda p:p.stat().st_mtime),state)

# Pobranie kolektora z GitHub
url = "https://raw.githubusercontent.com/danieldziezak1986-hash/weex-futures/main/weex_v651_collector.py"
r = requests.get(url, timeout=30)
r.raise_for_status()
Path("weex_v651_collector.py").write_bytes(r.content)

# Test wbudowany i skan TOP 150 + 50
%run weex_v651_collector.py --self-test
%run weex_v651_collector.py --top 150 --tradfi-top 50 --max-deep 30 --limit-ttl-minutes 180 --state-path /content/drive/MyDrive/WEEX/oi_history_shared.json

# Automatyczne pobranie najnowszego ZIP
folder = Path("/content/weex_v651_results")
archiwa = list(folder.glob("weex_v651_*.zip"))
if not archiwa:
    raise FileNotFoundError("Nie znaleziono ZIP WEEX V6.51")
najnowszy = max(archiwa, key=lambda p:p.stat().st_mtime)
print("Pobieranie wynikow:",najnowszy.name)
files.download(str(najnowszy))
